# Encoder parameter evaluation — baseline (2026-05-07)

Evaluates how accurately the encoder recovers the 25 physiological parameters from waveforms.

Ground-truth waveforms → encoder → predicted params (z-scored) → un-z-score → compare to GT params.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from dataset import CVDataset
from encoder import CVEncoder

In [ ]:
ENCODER_CKPT = ROOT / 'checkpoints/autoenc-baseline/checkpoint_017.pt'
STATS_PATH   = ROOT / 'norm_stats.json'
DATA_DIR     = Path('/media/8TBNVME/data/neh10/hdf5/cv8/simset_10M_cv8Eed_20260314/test')
MANIFEST     = DATA_DIR.parent / 'manifest_test.json'

BATCH_SIZE   = 256   # sims per batch
BATCH_IDX    = 0     # 0 → sims 0-255, 1 → sims 256-511, etc.

PARAM_NAMES = [
    'AVD', 'Bla', 'Blv', 'Bra', 'Brv', 'Cas', 'Cvp', 'Cvs',
    'Eap', 'Eedref_la', 'Eedref_lv', 'Eedref_ra', 'Eedref_rv',
    'Emax_LA', 'Emax_LV', 'Emax_RA', 'Emax_RV',
    'HR', 'Rap', 'Ras', 'Tmax', 'Tmax_a', 'Vs', 'τ', 'τ_a',
]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load encoder

In [ ]:
enc_ckpt = torch.load(ENCODER_CKPT, map_location=device)

encoder = CVEncoder().to(device)
encoder.load_state_dict(enc_ckpt['model_state'])
encoder.eval()

print(f"Encoder  epoch {enc_ckpt['epoch']}  val_loss={enc_ckpt['val_loss']:.6f}")

with open(STATS_PATH) as f:
    stats = json.load(f)
with open(MANIFEST) as f:
    manifest = json.load(f)

param_means = np.array([stats['parameters'][n]['mean'] for n in PARAM_NAMES])
param_stds  = np.array([stats['parameters'][n]['std']  for n in PARAM_NAMES])

## Run inference on test batch

In [ ]:
start = BATCH_IDX * BATCH_SIZE
end   = start + BATCH_SIZE
batch_entries = manifest['index'][start:end]
print(f'Batch {BATCH_IDX}: sims {start}–{end - 1}  ({len(batch_entries)} entries)')

ds     = CVDataset(DATA_DIR, batch_entries, stats)
loader = DataLoader(ds, batch_size=256, num_workers=0)

all_pred_params, all_gt_params = [], []
with torch.no_grad():
    for gt_params_b, waves_cont_b, waves_valve_b in loader:
        pred_params_b = encoder(waves_cont_b.to(device), waves_valve_b.to(device))
        all_pred_params.append(pred_params_b.cpu())
        all_gt_params.append(gt_params_b)
ds.close()

pred_z = torch.cat(all_pred_params).numpy()   # (N, 25) — z-scored
gt_z   = torch.cat(all_gt_params).numpy()     # (N, 25) — z-scored

# Un-z-score to physical units
pred_phys = pred_z * param_stds[None, :] + param_means[None, :]
gt_phys   = gt_z   * param_stds[None, :] + param_means[None, :]
print(f'Inference done: {pred_phys.shape[0]} sims, {pred_phys.shape[1]} parameters')

## Compute metrics

In [ ]:
abs_err  = np.abs(pred_phys - gt_phys)                          # (N, 25)
ch_mae   = abs_err.mean(axis=0)                                 # (25,)
ch_med   = np.median(abs_err, axis=0)                           # (25,)
ch_mape  = (abs_err / (np.abs(gt_phys) + 1e-9) * 100).mean(axis=0)   # (25,)

print(f"{'Parameter':<14} {'MAE':>12} {'Median AE':>12} {'MAPE (%)':>10}")
print('-' * 52)
for i, name in enumerate(PARAM_NAMES):
    print(f'{name:<14} {ch_mae[i]:>12.4f} {ch_med[i]:>12.4f} {ch_mape[i]:>10.2f}%')

## Error distributions

In [ ]:
x = np.arange(len(PARAM_NAMES))
w = 0.4
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(x, ch_mae, color='steelblue', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(PARAM_NAMES, rotation=45, ha='right')
axes[0].set_title('Mean MAE per parameter')
axes[0].set_ylabel('AE (physical units)')

axes[1].bar(x, ch_med, color='darkcyan', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(PARAM_NAMES, rotation=45, ha='right')
axes[1].set_title('Median AE per parameter')
axes[1].set_ylabel('AE (physical units)')

axes[2].bar(x, ch_mape, color='tomato', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(PARAM_NAMES, rotation=45, ha='right')
axes[2].set_title('MAPE per parameter')
axes[2].set_ylabel('MAPE (%)')

fig.suptitle(f'Encoder param errors — epoch {enc_ckpt["epoch"]}, batch {BATCH_IDX} (sims {start}–{end-1})', fontsize=13)
plt.tight_layout()
plt.show()

## Predicted vs GT scatter — all 25 parameters

In [ ]:
ncols = 5
nrows = 5
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

for i, name in enumerate(PARAM_NAMES):
    ax = axes[i // ncols, i % ncols]
    gt_i   = gt_phys[:, i]
    pred_i = pred_phys[:, i]
    lo, hi = gt_i.min(), gt_i.max()
    ax.scatter(gt_i, pred_i, s=10, alpha=0.5, color='steelblue')
    ax.plot([lo, hi], [lo, hi], color='tomato', linewidth=1, linestyle='--')  # y=x
    ax.set_title(f'{name}  (MAPE {ch_mape[i]:.1f}%)', fontsize=9)
    ax.set_xlabel('GT', fontsize=8)
    ax.set_ylabel('Pred', fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle(f'Encoder — predicted vs GT params  (epoch {enc_ckpt["epoch"]}, batch {BATCH_IDX})', fontsize=13)
plt.tight_layout()
plt.show()